# 05 — Train the models (experiments E1–E6)

**Goal:** train one or more models from the experiment plan and save each one so notebooks 03 and 04 can score and inspect it.

**The experiment plan.** Every experiment is a small settings file in `configs/experiments/`. They all use the same
training code and the same data; each one changes **one thing**, so we can see what that one thing does:

| Experiment | Settings file | What changes | Question it answers |
|---|---|---|---|
| E1 | `01_baseline_cnn` | small CNN trained from zero | how good is a network with no prior knowledge? |
| E2 | `02_effnet_frozen` | EfficientNet-B0, pretrained part **frozen** | are features learned on everyday photos enough? |
| E3 | `03_effnet_finetune` | EfficientNet-B0, **every layer learns** (main model) | does fine-tuning everything help? |
| E4 | `04_effnet_gradual` | like E3, but frozen for the first epoch | does unfreezing slowly help? |
| E5 | `05_resnet50_finetune` | like E3, but **ResNet-50** | does the architecture matter? |
| E6 | `06_effnet_clean` | like E3, but **clean photos only** (control) | does training with damaged photos help? |

You choose which ones to train in **Step 2** with `EXPERIMENTS`. **Each experiment is saved in its own folder**
named after it (e.g. `outputs/03_effnet_finetune/`), so training one never overwrites another.

**Labels:** `fake` = 1 (what we want to catch), `real` = 0. Every model outputs `P(fake)`, a number from 0 to 1.

**How to use:** this notebook is made for **Kaggle**. Do the settings below once, then run the cells from top to
bottom (Shift+Enter). Training only starts when you switch it on (Step 7).

### Before you start: Kaggle settings

Open the right-hand panel of the notebook editor (the **>|** icon at the top right if it is hidden):

- **Settings → Internet → On.** Step 1 downloads the project code from GitHub. Kaggle asks you to verify your phone once.
- **Add Input → Datasets →** search **"140k Real and Fake Faces"** (by xhlulu) → **+**. These are the images.
  Kaggle attaches them read-only under `/kaggle/input/`, so nothing is downloaded or unzipped.
- **Settings → Accelerator → GPU T4 x2.** Makes the model steps much faster (free, about 30 GPU hours a week).
- **To keep the trained models:** choose `EXPERIMENTS` (Step 2) and set `RUN_TRAINING = True` (Step 7), then
  click **Save Version → Save & Run All (Commit)**. Kaggle runs the whole notebook again from the top and keeps
  `/kaggle/working/`, including `outputs/`. Notebooks 03 and 04 then use it with **Add Input → Your Work**.

After you change a setting, Kaggle restarts the session: run the cells again from the top.

## The models at a glance: original vs. our changes

**What we start from.** E2–E4 and E6 use **EfficientNet-B0** (the *architecture*: the network's design, about
4 million weights) **pretrained on ImageNet** (the *dataset* its starting weights learned from: 1.2 million everyday
photos in 1,000 classes such as dogs, cars and cups). The weights come from the `timm` library, in one line of
`src/deepfake_kyc/models.py`:

```python
timm.create_model("efficientnet_b0", pretrained=True, num_classes=1, drop_rate=0.2)
```

- `pretrained=True` downloads the ImageNet weights (about 20 MB, only the first time).
- `num_classes=1` **throws away the original 1,000-class last layer** and puts a **new, random 1-output layer** in its place.
- `drop_rate=0.2` adds dropout (randomly ignores 20% of the features while training) just before that new layer.

### Inside the EfficientNet-B0 backbone

```
photo 224×224×3
   │
   ▼
stem: one 3×3 convolution              → 112×112×32    finds edges and colours
   │
   ▼
16 MBConv blocks in 7 stages           → 7×7×320       each stage looks at bigger patterns:
   │                                                   edges → textures → parts (eyes, skin) → whole objects
   ▼
1×1 convolution                        → 7×7×1,280     1,280 "pattern detectors" for each of 7×7 image regions
   │
   ▼
global average pooling                 → 1,280         one number per detector, averaged over the whole photo
```

An **MBConv** block is EfficientNet's building block: it widens the features, filters each channel cheaply,
re-weights the channels by importance ("squeeze-and-excitation"), and narrows them again. "Efficient" means it gets
high accuracy with few weights (about 4 million, vs. about 138 million in VGG16 and 25 million in ResNet-50).

### Original EfficientNet-B0 (as downloaded)

```
photo 224×224×3
   │
   ▼
backbone: 4,007,548 weights          learned on ImageNet
   │  pooling → 1,280 features
   ▼
last layer: 1,280 → 1,000            1,281,000 weights   ✗ thrown away
   │
   ▼
softmax → "which of 1,000 objects?"
```

### E2 — frozen backbone, train only the new last layer

```
face 224×224×3
   │
   ▼
backbone: 4,007,548 weights          🔒 FROZEN: never changes (BatchNorm frozen too)
   │  pooling → 1,280 features
   ▼
NEW last layer: 1,280 → 1            1,281 weights       ✏️ the only part that learns (learning rate 0.001)
   │
   ▼
sigmoid → P(fake)
```

### E3 — fine-tune everything (main model)

```
face 224×224×3
   │
   ▼
backbone: 4,007,548 weights          ✏️ learns slowly (learning rate 0.00003 = 10x slower), BatchNorm adapts to faces
   │  pooling → 1,280 features
   ▼
NEW last layer: 1,280 → 1            1,281 weights       ✏️ learns (learning rate 0.0003)
   │
   ▼
sigmoid → P(fake)
```

**Where do the 1,281 numbers come from?** The new last layer turns the 1,280 features into one score:
`score = w₁·f₁ + w₂·f₂ + … + w₁₂₈₀·f₁₂₈₀ + b`, then `P(fake) = sigmoid(score)`.
That is **1,280 weights** (one per feature) **+ 1 bias** = 1,281 — the same rule as a Keras `Dense` layer:
inputs × outputs + outputs. The original layer scored 1,000 classes: 1,280 × 1,000 + 1,000 = 1,281,000.

**What E2 and E3 change, and where:**

| | E2 (`02_effnet_frozen.yaml`) | E3 (`03_effnet_finetune.yaml`) |
|---|---|---|
| backbone | frozen for the whole run (`freeze_backbone_epochs: 999`) | learns from the start (`freeze_backbone_epochs: 0`, the default) |
| BatchNorm | frozen too, so the features never drift | updates its averages to fit faces |
| learning rate | 0.001 for the last layer (`lr: 1.0e-3`) | 0.0003 for the last layer, 0.00003 for the backbone (defaults) |
| weights that learn | 1,281 | 4,008,829 (all) |
| question | are ImageNet features enough on their own? | does adapting the backbone to deepfakes help? |

Freezing happens in `src/deepfake_kyc/train.py` (`set_backbone_trainable`, every epoch) and
`src/deepfake_kyc/engine.py` (`freeze_batch_norm`). Everything else (data, damage on half the training photos,
5 epochs, batch size 64, AdamW, warm-up then cosine learning-rate schedule, dropout 0.2, seed 42) is the same.

**The other experiments** use the same shape: **E1** is a small CNN trained from zero (no pretraining),
**E4** is E3 but frozen during epoch 1, **E5** is E3 with **ResNet-50** (also pretrained on ImageNet) instead of
EfficientNet-B0, and **E6** is E3 trained on clean photos only.

**In the report, write:** *"E2 and E3 use an EfficientNet-B0 model pretrained on ImageNet (weights from the `timm`
library), with its 1,000-class classifier replaced by a single-output real/fake head. E2 freezes the pretrained
backbone and trains only the new head; E3 fine-tunes all layers, with the backbone at a 10× lower learning rate."

## Step 1 — Get the project and the tools ready

**What this does, in plain words:**

1. Loads the Python tools we need.
2. Downloads the project code (`src/`, `configs/`) from GitHub into `/kaggle/working/`, the only folder on Kaggle
   where we can write files. If Internet is off, it tells you so in plain words.
3. Finds the dataset you attached with **Add Input** somewhere under `/kaggle/input/`.
4. Installs any missing package the training code needs (Kaggle already has PyTorch, so it is never reinstalled).
5. Tells you whether a GPU is switched on.

In [ ]:
# 1. Tools we need
import importlib.util
import json
import os
import random
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image

# 2. Get the project code (src/ and configs/) from GitHub.
#    It goes into /kaggle/working: the only folder on Kaggle where we can write files.
REPO_URL = "https://github.com/msondhi/UOT_SCS_3546_041_Deep-Learning.git"
PROJECT_ROOT = Path("/kaggle/working/UOT_SCS_3546_041_Deep-Learning")

if not PROJECT_ROOT.exists():
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)])
    if result.returncode != 0:
        raise RuntimeError("Could not download the project from GitHub. "
                           "Turn on Settings -> Internet in the right-hand panel, then run this cell again.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))   # lets Python import the project's own code
print("Project folder:", PROJECT_ROOT)

# 3. Find the dataset you attached with Add Input. Kaggle puts it somewhere under /kaggle/input.
def find_dataset():
    """Look inside /kaggle/input for the folder called real-vs-fake that has train/ valid/ test/."""
    for depth in range(1, 5):                                  # look 1, 2, 3, then 4 folders deep
        pattern = "/".join(["*"] * depth) + "/real-vs-fake"    # e.g. "*/*/real-vs-fake"
        for folder in Path("/kaggle/input").glob(pattern):
            if (folder / "train").is_dir():
                return folder
    return None

DATA_ROOT = find_dataset()       # folder with train/ valid/ test/, or None if not attached
if DATA_ROOT is None:
    print('Dataset not found: Add Input -> Datasets -> "140k Real and Fake Faces", then run this cell again.')
else:
    print("Images folder:", DATA_ROOT)

# 4. Install any missing package (Kaggle already has PyTorch, so it is never reinstalled).
# {name used in "import": name used by pip}
needed = {"timm": "timm", "sklearn": "scikit-learn", "yaml": "PyYAML", "tqdm": "tqdm"}
for module, package in needed.items():
    if importlib.util.find_spec(module) is None:
        print("Installing", package, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

# 5. GPU or not?
import torch

print("PyTorch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: training will be very slow. Settings -> Accelerator -> GPU T4 x2, then run again.")

## Step 2 — Choose the experiment(s) and read their settings

**What this does:** reads the settings file of every experiment in `EXPERIMENTS` and prints the important settings
in plain words, so you know exactly what will be trained. Each file only lists what it changes; everything else
comes from `configs/base.yaml`, shared by all experiments.

**Choose here** which experiments to train in this run, for example:

```python
EXPERIMENTS = ["02_effnet_frozen"]                                             # one experiment
EXPERIMENTS = ["03_effnet_finetune", "04_effnet_gradual", "06_effnet_clean"]   # several, one after another
```

**No overwriting:** each experiment saves into its own folder named after it. Step 7 also refuses to train an
experiment again if it already has a saved model, unless you switch on `OVERWRITE`.

**On Kaggle, train all the experiments you want in one run.** A new **Save Version** keeps only the files of
that run, and notebooks 03 and 04 attach one version of a notebook. (Or make one copy of this notebook per experiment.)

In [ ]:
from deepfake_kyc.config import load_config

# The experiment plan: settings file name -> what it tests
EXPERIMENT_PLAN = {
    "01_baseline_cnn":      "E1: small CNN trained from zero (no pretrained knowledge)",
    "02_effnet_frozen":     "E2: EfficientNet-B0, pretrained part frozen, only the last layer learns",
    "03_effnet_finetune":   "E3: EfficientNet-B0, every layer learns (main model)",
    "04_effnet_gradual":    "E4: like E3, but the pretrained part is frozen for the first epoch",
    "05_resnet50_finetune": "E5: like E3, but ResNet-50 instead of EfficientNet-B0",
    "06_effnet_clean":      "E6: like E3, but trained on clean photos only (the control)",
}

# Which experiment(s) to train in this run. Each one is saved in its own folder, so they never overwrite each other.
EXPERIMENTS = ["02_effnet_frozen"]      # e.g. ["03_effnet_finetune", "04_effnet_gradual", "06_effnet_clean"]


def training_style(settings):
    """Describe in words how much of the model learns."""
    epochs = settings["train"]["epochs"]
    frozen_epochs = settings["train"]["freeze_backbone_epochs"]
    slower = 1 / settings["train"]["backbone_lr_mult"]
    if not settings["model"]["pretrained"]:
        return "the whole network learns from zero"
    if frozen_epochs >= epochs:
        return "pretrained part frozen for the whole run, only the last layer learns"
    if frozen_epochs > 0:
        return f"pretrained part frozen for the first {frozen_epochs} epoch(s), then everything learns"
    return f"everything learns from the start (pretrained part {slower:g}x slower than the last layer)"


configs = {}     # experiment name -> its settings
for name in EXPERIMENTS:
    if name not in EXPERIMENT_PLAN:
        raise ValueError(f"Unknown experiment {name!r}. Choose from: {list(EXPERIMENT_PLAN)}")
    settings = load_config(PROJECT_ROOT / "configs" / "experiments" / f"{name}.yaml")
    configs[name] = settings

    print("=" * 70)
    print(name, "-", EXPERIMENT_PLAN[name])
    print("   model:      ", settings["model"]["name"], "(pretrained on ImageNet)" if settings["model"]["pretrained"] else "(no pretraining)")
    print("   training:   ", training_style(settings))
    print("   epochs:     ", settings["train"]["epochs"], "| batch size:", settings["data"]["batch_size"],
          "| learning rate:", settings["train"]["lr"])
    print("   early stop: ", "if the validation loss does not improve for",
          settings["train"]["early_stopping_patience"], "epochs in a row (the best epoch is kept)")
    print("   trained on: ", "real + fake faces, half randomly damaged (blur, JPEG, ...)"
          if settings["augment"]["degradation"] else "real + fake faces, clean only")
    print("   saved in:    outputs/" + settings["experiment_name"] + "/")

# Settings shared by every experiment (image folder, image size, random seed) are read from the first one.
cfg = configs[EXPERIMENTS[0]]
print("=" * 70)
print("Random seed:", cfg["seed"], "(same seed = same result every run)")

## Step 3 — Check the dataset

**What this does:** counts the images in `train`, `valid` and `test` of the dataset you attached with
**Add Input** (see the settings box at the top), and prints `Dataset ready.`

The first count reads each folder once (a few seconds per 50,000 files); the lists are then remembered,
so later steps are instant.

In [ ]:
SPLITS = ["train", "valid", "test"]
IMAGE_LISTS = {}   # remembers the file list of each folder, so every folder is read only once

def list_images(split, class_name):
    """All .jpg files in one folder, e.g. list_images("train", "fake").

    The first call reads the folder (a few seconds for 50,000 files); after that the list
    is remembered in IMAGE_LISTS and every later call is instant.
    """
    if DATA_ROOT is None:
        return []
    folder = DATA_ROOT / split / class_name
    if folder not in IMAGE_LISTS:
        names = sorted(os.listdir(folder)) if folder.is_dir() else []    # sorting plain names is fast
        IMAGE_LISTS[folder] = [folder / name for name in names if name.lower().endswith(".jpg")]
    return IMAGE_LISTS[folder]

def data_ready():
    """True if every split has real and fake images."""
    return all(list_images(split, "real") and list_images(split, "fake") for split in SPLITS)

for split in SPLITS:
    print(f"{split:6} real={len(list_images(split, 'real')):,}  fake={len(list_images(split, 'fake')):,}")
DATA_AVAILABLE = data_ready()
if DATA_AVAILABLE:
    print("Dataset ready.")
else:
    print('Dataset missing: Add Input -> Datasets -> "140k Real and Fake Faces", then run Step 1 again.')

## Step 4 — Where the trained models are saved

**What this does:** chooses the folder for the results: the project's `outputs/` folder inside `/kaggle/working/`.
Each experiment gets its own sub-folder, e.g. `outputs/03_effnet_finetune/`, so they never overwrite each other.

**Important:** `/kaggle/working/` is kept only when you click **Save Version → Save & Run All (Commit)**.
If you just close the session, the trained models are gone.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"        # /kaggle/working/UOT_SCS_3546_041_Deep-Learning/outputs
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Results will be saved in:", OUTPUT_DIR)

## Step 5 — A quick look at the training photos

**What this does:** shows 3 real and 3 fake training faces. A quick check that the images and labels look right
(notebook 01 has the full data exploration).

In [ ]:
import random

if DATA_AVAILABLE:
    random.seed(cfg["seed"])
    examples = [(path, "real (0)") for path in random.sample(list_images("train", "real"), 3)] + \
               [(path, "fake (1)") for path in random.sample(list_images("train", "fake"), 3)]

    fig, axes = plt.subplots(1, 6, figsize=(15, 3))
    for ax, (path, label) in zip(axes, examples):
        with Image.open(path) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(label)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: dataset not available (Step 3).")

## Step 6 — Build each model and check it

**What this does, for every experiment in `EXPERIMENTS`:**

1. Builds the model (the first time, `timm` downloads the pretrained weights: about 20 MB for EfficientNet-B0,
   about 100 MB for ResNet-50).
2. Freezes the pretrained part if that experiment starts frozen (E2, E4), exactly like the training code does.
3. Counts how many numbers (parameters) the model has and how many learn in the first epoch.
4. Pushes 2 blank test images through it to check the output shape: one number per photo.
   This only checks the wiring; it is not a result.

Each model is deleted again after the check, so it does not take up memory needed for training.

In [ ]:
from deepfake_kyc.models import build_model, set_backbone_trainable, split_parameters
from deepfake_kyc.utils import get_device, seed_everything

seed_everything(cfg["seed"])     # same random numbers every run
device = get_device()            # GPU if there is one
print("Running on:", device)

count = lambda params: sum(p.numel() for p in params)

for name, settings in configs.items():
    model = build_model(settings["model"]["name"], pretrained=settings["model"]["pretrained"],
                        dropout=settings["model"]["dropout"]).to(device)
    frozen_at_start = settings["model"]["pretrained"] and settings["train"]["freeze_backbone_epochs"] >= 1
    set_backbone_trainable(model, trainable=not frozen_at_start)

    pretrained_part, last_layer = split_parameters(model)
    learning = count(p for p in model.parameters() if p.requires_grad)

    size = settings["data"]["image_size"]
    blank_photos = torch.zeros(2, 3, size, size, device=device)    # 2 photos, 3 colours, size x size
    model.eval()
    with torch.no_grad():
        output = model(blank_photos)

    print("=" * 70)
    print(name)
    print(f"   all parameters:            {count(model.parameters()):,}")
    print(f"   feature extractor:         {count(pretrained_part):,}" + ("  (frozen at the start)" if frozen_at_start else ""))
    print(f"   last layer:                {count(last_layer):,}")
    print(f"   learning in epoch 1:       {learning:,}")
    print("   output shape:", tuple(output.shape), "-> one number per photo (expected (2, 1))")

    del model                     # free the memory before the next model / training
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Step 7 — Train (switch on when ready)

**What happens during training (for each experiment, one after another):**

1. The model sees a batch of training photos (half of them randomly damaged, except E6) and guesses `P(fake)` for each.
2. The **loss** (binary cross-entropy) measures how wrong the guesses were.
3. The optimiser (AdamW) nudges the model's numbers to make the loss smaller. Repeat for every batch = one **epoch**.
4. After each epoch the model is scored on the **validation** photos (never damaged, never used for learning).
5. The epoch with the lowest validation loss is saved as `best.pt`. Training stops early if validation stops improving.

**Controls:**
- `RUN_TRAINING = True` starts it.
- `QUICK_RUN = True`: a short test run per experiment (1,000 training photos, 300 validation photos, 1 epoch,
  a few minutes). It is saved as `<experiment>_notebook_demo`, so it never mixes with a real run
  (notebooks 03 and 04 will still list it; just ignore it when comparing experiments).
- `QUICK_RUN = False`: the real run with all 100,000 training photos, saved as `<experiment>` (the same name
  `train.py` uses). On a T4 GPU expect roughly up to an hour per experiment; E1 and E2 are faster.
- `OVERWRITE = False`: an experiment that already has a saved model in its folder is **skipped**, so a finished
  model is never replaced by accident. Set `True` only if you really want to train it again.

**Speed settings (already on, nothing to do):**
- all of Kaggle's CPU cores (4) prepare the photos in parallel, so the GPU does not wait for them;
- **mixed precision** (`train.amp: true` in `configs/base.yaml`): the GPU computes in 16-bit where it is safe,
  about 2x faster with the same accuracy. Each training run prints `Mixed precision: on` to confirm;
- each epoch's time is printed (`seconds=...`) and saved in `history.csv`.

Kaggle's "GPU T4 x2" gives two GPUs, but training uses one of them; a single P100 works just as well.

**Kaggle limits:** one run can last at most 12 hours, and the free GPU allows about 30 hours a week,
so train two or three full experiments per run at most.

The test photos are **not** used here; they are kept for the final score in Step 9.

In [ ]:
RUN_TRAINING = False     # change to True to train
QUICK_RUN = True         # True = short test run; False = the full run
OVERWRITE = False        # True = train again even if this experiment already has a saved model

RUN_DIRS = {}            # experiment name -> folder where its model was saved

if RUN_TRAINING and not DATA_AVAILABLE:
    print("Cannot train: the dataset is missing (Step 3).")
elif RUN_TRAINING:
    from deepfake_kyc.train import main as run_training

    for name, settings in configs.items():
        run_name = f"{name}_notebook_demo" if QUICK_RUN else settings["experiment_name"]
        run_dir = OUTPUT_DIR / run_name
        print("=" * 70)
        print("Experiment:", name, "-", EXPERIMENT_PLAN[name])

        # Never replace a finished model by accident.
        if (run_dir / "best.pt").exists() and not OVERWRITE:
            print("Skipped: a model is already saved in", run_dir, "(set OVERWRITE = True to train again).")
            RUN_DIRS[name] = run_dir
            continue

        # Changes on top of the settings file, written as "setting=value" like in a terminal.
        # data.root = where the images are (the attached dataset under /kaggle/input).
        # data.num_workers = how many CPU cores prepare photos in parallel (Kaggle has 4), so the GPU never waits.
        changes = [f"output_dir={OUTPUT_DIR.as_posix()}", f"data.num_workers={os.cpu_count()}",
                   f"data.root={DATA_ROOT.as_posix()}",
                   f"experiment_name={run_name}"]
        if QUICK_RUN:
            changes += ["data.train_subset=1000", "data.val_subset=300", "train.epochs=1"]

        config_file = PROJECT_ROOT / "configs" / "experiments" / f"{name}.yaml"
        run_training(["--config", str(config_file), "--set", *changes])
        RUN_DIRS[name] = run_dir
        print("Saved in:", run_dir)
else:
    print("Skipped. Change RUN_TRAINING to True to train.")

## Step 8 — Read the training results

**What this does:** for every experiment in `EXPERIMENTS`, opens the files its training saved and shows them:

| File | What is inside |
|---|---|
| `best.pt` | the trained model (used by notebooks 03 and 04) |
| `config.yaml` | the exact settings of this run |
| `history.csv` | one row per epoch: training loss, validation loss and validation scores |
| `summary.json` | which epoch was best and its scores |

Then a small table compares the best validation scores of all the experiments, and one chart per experiment
shows the loss per epoch.

**How to read the chart:** both losses should go down. If the training loss keeps falling but the validation loss
goes up, the model is starting to memorise the training photos (overfitting).

These are **validation** scores. Use them to choose settings; the test score comes last (Step 9).

In [ ]:
def find_run_dir(name):
    """The folder of this experiment's newest finished run (full run or quick demo), or None."""
    runs = [OUTPUT_DIR / run_name for run_name in [name, f"{name}_notebook_demo"]
            if (OUTPUT_DIR / run_name / "summary.json").exists()]
    return max(runs, key=lambda folder: (folder / "summary.json").stat().st_mtime) if runs else None

# Use the folders from Step 7; for experiments trained earlier, look for them.
for name in EXPERIMENTS:
    if name not in RUN_DIRS or not (RUN_DIRS[name] / "summary.json").exists():
        RUN_DIRS[name] = find_run_dir(name)

best_rows = []
for name in EXPERIMENTS:
    run_dir = RUN_DIRS[name]
    if run_dir is None:
        print(name, ": no finished run yet in", OUTPUT_DIR, "-> train it in Step 7.")
        continue
    summary = json.loads((run_dir / "summary.json").read_text())
    history = pd.read_csv(run_dir / "history.csv")
    best = summary["best"]
    best_rows.append({"run": run_dir.name, "best epoch": summary["best_epoch"],
                      "val_loss": best["val_loss"], "val_accuracy": best["val_accuracy"],
                      "val_f1": best["val_f1"], "val_roc_auc": best["val_roc_auc"]})

    plt.figure(figsize=(6, 3.5))
    plt.plot(history["epoch"], history["train_loss"], marker="o", label="training loss")
    plt.plot(history["epoch"], history["val_loss"], marker="o", label="validation loss")
    plt.xlabel("epoch")
    plt.ylabel("loss (lower is better)")
    plt.title(f"{run_dir.name}: loss per epoch")
    plt.legend()
    plt.show()

if best_rows:
    print("Best validation scores (one row per experiment):")
    display(pd.DataFrame(best_rows).round(4))

## Step 9 (once, at the very end) — Score the models on the test photos

**Best practice:** look at the test photos **only once**, after you have finished choosing settings with the
validation scores. If you change settings because of the test score, the test score is no longer honest.

**What this does:** for every experiment in `EXPERIMENTS` that has a trained model, runs `evaluate.py` on the test
photos and saves `eval_test.csv` in **that experiment's own folder**, next to its `best.pt`, exactly like
notebook 03, Step 1b. Each model is tested the way it was trained: models trained on clean + damaged photos are
also tested on damaged photos (26 tests, slower); E6 (clean only) gets the clean test.

**Quick test:** with `QUICK_RUN = True` (Step 7) each test uses **2,000** random test photos instead of all 20,000,
so all 26 tests finish in a few minutes. A full run (`QUICK_RUN = False`) uses all 20,000 for the real final score.
Quick-run models live in their own `<experiment>_notebook_demo` folders, so a quick test never blocks the real test.

If an experiment already has `eval_test.csv` (for example from notebook 03), it is not scored again.
Notebook 03 then compares all the experiments.

In [ ]:
RUN_TEST = False         # change to True for the final test score
# How many test photos: a quick run (Step 7) gets a quick test of 2,000 photos; a full run uses all 20,000.
TEST_PHOTOS = 2000 if QUICK_RUN else None   # None = all 20,000 test photos (the real final score)

if RUN_TEST:
    from deepfake_kyc.evaluate import main as run_evaluate

    for name in EXPERIMENTS:
        run_dir = RUN_DIRS.get(name)
        print("=" * 70)
        print("Experiment:", name)
        if run_dir is None or not (run_dir / "best.pt").exists():
            print("No trained model found: train it in Step 7 first.")
            continue
        if (run_dir / "eval_test.csv").exists():
            print("Already scored:", run_dir / "eval_test.csv", "-> see the results in notebook 03.")
            continue

        arguments = ["--checkpoint", str(run_dir / "best.pt"), "--set", f"data.num_workers={os.cpu_count()}",
                     f"data.root={DATA_ROOT.as_posix()}"]     # where the test images are on this computer
        if TEST_PHOTOS is not None:
            arguments.append(f"data.test_subset={TEST_PHOTOS}")
        if configs[name]["augment"]["degradation"]:          # trained with damage -> also test with damage
            arguments += ["--degradation", "all"]
        rows = run_evaluate(arguments)
        display(pd.DataFrame(rows).round(3))
else:
    print("Skipped. Change RUN_TEST to True only when you have finished choosing settings.")

## What comes next?

- **Compare:** run [03_results_charts.ipynb](03_results_charts.ipynb) to compare all trained experiments,
  and [04_error_analysis.ipynb](04_error_analysis.ipynb) to see which photos a model gets wrong.
  On Kaggle, attach this notebook to them with **Add Input → Your Work → Notebooks**.
- **More experiments:** add their names to `EXPERIMENTS` in Step 2 and run Steps 2–9 again. Models that are
  already saved are skipped, so nothing is lost.
- **Fair comparison:** every experiment uses the same seed, the same data and the same training code; only the
  settings file differs. Keep it that way: change settings in the files in `configs/experiments/`, not by hand here.

**Limits and responsible use:** these research models tell apart the kind of photos in this dataset. They do not
check who a person is and do not prove fraud. Results may not hold for other AI generators, cameras or groups of
people. Never use them as the only basis for a real decision, and never use private customer photos.